# NB6 — MIN2: regenerate data → train scorer → evaluate LOO

Experiment contract:

```text
Compatibility scorer: 2–8 items
LOO diagnosis: original outfit >= 3 items
```

Notebook chạy trên branch `exp/min2-scorer-loo3`, giữ frozen V2 nguyên vẹn và ghi artifact vào experiment directories.

In [ ]:
from pathlib import Path
import json, subprocess, sys
import torch
from torch.utils.data import DataLoader
from functools import partial

ROOT = Path.cwd().resolve()
branch = subprocess.check_output(["git","rev-parse","--abbrev-ref","HEAD"], text=True).strip()
assert branch == "exp/min2-scorer-loo3", branch
sys.path.insert(0, str(ROOT))

from src.data.runtime_paths import load_runtime_paths
from src.data.min2_experiment import (
    EXPERIMENT_DATASET_VERSION, EXPERIMENT_TAG,
    MIN_SCORER_ITEMS, MAX_SCORER_ITEMS, LOO_MIN_ORIGINAL_ITEMS,
    prepare_min2_positives, validate_min2_embeddings,
    build_min2_scorer_dataset, scorer_ready_path,
)
from src.scorer.min2_experiment import (
    load_config, build_train_valid_loaders_min2, build_min2_datasets,
    build_min2_provenance, fit_min2_scorer, collate_min2_scorer_batch,
)
from src.scorer.model import TypeAwarePairwiseScorer
from src.scorer.train import seed_everything
from src.scorer.evaluate import evaluate_model
from src.scorer import checkpoint as checkpoint_utils
from src.scorer.dataset import read_jsonl
from src.diagnosis.evaluate_loo import evaluate_loo_dataset

In [ ]:
PATHS_CONFIG = ROOT / "configs/data_paths.min2_experiment.json"
SCORER_CONFIG = ROOT / "configs/scorer_type_aware_pairwise_min2_experiment.yaml"
MAPPING = ROOT / "configs/category_mapping_core7_v2.json"

paths = load_runtime_paths(repo_root=ROOT, config_path=PATHS_CONFIG)
config = load_config(SCORER_CONFIG)

print("dataset:", EXPERIMENT_DATASET_VERSION)
print("scorer min/max:", MIN_SCORER_ITEMS, MAX_SCORER_ITEMS)
print("LOO min original:", LOO_MIN_ORIGINAL_ITEMS)
print("core7:", paths.core7_dir)
print("scorer-ready:", paths.scorer_ready_dir)
print("embedding cache:", paths.embedding_cache)

## 1. Regenerate MIN2 dataset

Nếu embedding validation fail sau khi thêm outfit size 2, hãy mở rộng/rebuild FashionCLIP cache cho các item thiếu rồi chạy lại. Không bypass validation.

In [ ]:
DEBUG_LIMIT = None  # đặt 500 để smoke test; None = full official train/valid/test

prepare_report = prepare_min2_positives(
    paths, mapping_path=MAPPING, debug_limit=DEBUG_LIMIT
)
for split, report in prepare_report["splits"].items():
    print(split, report["outfits"]["outfits_kept"],
          report["outfits"]["outfit_length_distribution_after"])

embedding_report = validate_min2_embeddings(paths, mapping_path=MAPPING)
for split, report in embedding_report["splits"].items():
    print(split, "coverage=", report["embedding_coverage"],
          "missing=", report["missing_or_invalid_embedding_count"])
assert embedding_report["pass"], "Embedding coverage must PASS."

dataset_report = build_min2_scorer_dataset(
    paths, mapping_path=MAPPING, overwrite=True
)
print(json.dumps(dataset_report, indent=2))
assert dataset_report["status"] == "READY_TO_TRAIN"

In [ ]:
from collections import Counter

for split in ("train","valid","test"):
    rows = read_jsonl(scorer_ready_path(paths.scorer_ready_dir, split))
    lengths = Counter(len(r["items"]) for r in rows)
    print(split, "samples=", len(rows), "lengths=", dict(sorted(lengths.items())))
    assert lengths.get(2, 0) > 0, f"{split}: no size-2 samples"

## 2. Train Type-aware Pairwise scorer from scratch

Architecture và V5 optimization protocol giữ nguyên; lower bound duy nhất đổi sang 2 items.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_DIR = (
    paths.artifact_root / "checkpoints" /
    "type_aware_pairwise_v1" / "min2_exp_v1_seed42"
)

seed_everything(int(config["training"]["seed"]))
model = TypeAwarePairwiseScorer.from_config(config)
loaders = build_train_valid_loaders_min2(paths, config, num_workers=0)
provenance = build_min2_provenance(paths, ROOT)

train_result = fit_min2_scorer(
    model,
    loaders["train_loader"],
    loaders["valid_loader"],
    config=config,
    checkpoint_dir=CHECKPOINT_DIR,
    provenance=provenance,
    device=DEVICE,
)
print("best epoch:", train_result["best_epoch"])
print("best valid ROC-AUC:", train_result["best_valid_roc_auc"])
print("checkpoint:", train_result["best_checkpoint"])

## 3. Load best checkpoint and evaluate scorer

Set `EVALUATE_TEST=False` nếu muốn giữ test blind.

In [ ]:
best_model = TypeAwarePairwiseScorer.from_config(config)
ckpt = checkpoint_utils.load_checkpoint(
    train_result["best_checkpoint"], model=best_model, map_location="cpu"
)
best_model.to(DEVICE).eval()

EVALUATE_TEST = True
eval_splits = ["valid"] + (["test"] if EVALUATE_TEST else [])
eval_datasets, _ = build_min2_datasets(paths, splits=tuple(eval_splits))

scorer_metrics = {}
for split in eval_splits:
    loader = DataLoader(
        eval_datasets[split],
        batch_size=int(config["training"]["batch_size"]),
        shuffle=False,
        collate_fn=partial(collate_min2_scorer_batch, max_items=MAX_SCORER_ITEMS),
    )
    result = evaluate_model(best_model, loader, device=DEVICE)
    scorer_metrics[split] = result["metrics"]
    print("\nSCORER", split)
    print(json.dumps(result["metrics"], indent=2))

## 4. LOO diagnosis metrics

Theo `PROJECT_METRICS_VI.md`:

- **LOO Top-1 Localization Accuracy**: item có `delta_i = C(O \ x_i) - C(O)` lớn nhất phải trùng `swapped_item_index`.
- **LOO Hit@2**: ground-truth swapped item nằm trong 2 delta lớn nhất.

Chỉ synthetic negative có original outfit `>=3` được evaluate. Size-2 negatives vẫn dùng cho scorer nhưng bị skip khỏi LOO.

In [ ]:
loo_results = {}
for split in eval_splits:
    result = evaluate_loo_dataset(best_model, eval_datasets[split], device=DEVICE)
    loo_results[split] = result
    print("\nLOO", split)
    print(json.dumps(result["metrics"], indent=2))
    print("by length:")
    for row in result["by_length"]:
        print(row)

## 5. Save evaluation artifacts

In [ ]:
EVAL_DIR = paths.scorer_ready_dir / "evaluation_min2_exp_v1"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

summary = {
    "experiment": EXPERIMENT_TAG,
    "dataset_version": EXPERIMENT_DATASET_VERSION,
    "scorer_min_items": MIN_SCORER_ITEMS,
    "loo_min_original_items": LOO_MIN_ORIGINAL_ITEMS,
    "checkpoint": str(train_result["best_checkpoint"]),
    "checkpoint_epoch": int(ckpt["epoch"]),
    "scorer_metrics": scorer_metrics,
    "loo_metrics": {s: loo_results[s]["metrics"] for s in eval_splits},
    "loo_metrics_by_length": {s: loo_results[s]["by_length"] for s in eval_splits},
}

with (EVAL_DIR / "evaluation_summary.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
    f.write("\n")

for split in eval_splits:
    with (EVAL_DIR / f"loo_predictions_{split}.jsonl").open("w", encoding="utf-8") as f:
        for row in loo_results[split]["predictions"]:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(json.dumps(summary, indent=2))
print("saved to:", EVAL_DIR)

## Acceptance check

Trước khi promote MIN2 thành canonical:

1. scorer-ready data có size-2 samples và `READY_TO_TRAIN`;
2. ROC-AUC / 2-way FITB không giảm đáng kể trên comparable data;
3. LOO Top-1 cho original size 3 đủ tốt;
4. xem thêm LOO Hit@2 và breakdown theo outfit length;
5. original size 2 không được tính LOO.